## Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from textwrap import wrap
import numpy as np
import re
import os

#Helper Functions

def normalize_text(text):
    if not isinstance(text, str):
        return text
    text = text.lower().strip()                   # lowercase + trim
    text = re.sub(r"[^\w\s]", "", text)            # remove punctuation
    text = re.sub(r"\s+", " ", text)               # collapse multiple spaces
    return text



# Analyses

## Why do some MCP mappings have a 5 for Physical Tasks

This table shows that for the mcps that rated a physical task 3-5, they on average just were more liberal with higher ratings. We see that it it not due to language or length of the description, which means there must be something about these MCPs that makes them seem more applicable to the physical tasks, or that make them seem more applicable to tasks in general. 

In [1]:
import pandas as pd
import re
import numpy as np

# 1. Load data
mcp_df = pd.read_csv("../data/mcp_results_2026-02-18.csv")
task_ref = pd.read_csv("../data/task_results_2026-02-18_with_physical.csv")

# Create a set of normalized physical task names for fast lookup
def normalize(t):
    return re.sub(r"[^\w\s]", "", str(t).lower().strip())

physical_tasks = set(task_ref[task_ref['physical'] == True]['task'].apply(normalize))

def parse_and_analyze(row):
    """
    Parses the rating string. 
    Format: "Task Description (Occ Name): Score; Task Description..."
    """
    ratings_str = str(row['task_ratings'])
    # Split by semicolon to get individual task ratings
    parts = ratings_str.split(';')
    
    task_scores = []
    has_high_physical = False
    
    for part in parts:
        if ':' not in part: continue
        
        # Split at the last colon to separate Task(Occ) from Score
        sub_parts = part.rsplit(':', 1)
        if len(sub_parts) < 2: continue
        
        raw_task_info = sub_parts[0].strip()
        try:
            score = int(re.search(r'\d+', sub_parts[1]).group())
        except: continue
        
        # Extract task description: "Description (Occ Name)" -> "Description"
        # We look for the part before the last parentheses
        task_desc = re.sub(r'\s*\([^)]*\)$', '', raw_task_info).strip()
        norm_task = normalize(task_desc)
        
        task_scores.append(score)
        
        # Check if this specific task is Physical AND score is 3, 4, or 5
        if norm_task in physical_tasks and score >= 3:
            has_high_physical = True
            
    if not task_scores:
        return pd.Series([None] * 7 + [False], index=['mean', 'count_1', 'count_2', 'count_3', 'count_4', 'count_5', 'total_ratings', 'is_high_phys_mcp'])

    # Statistics for this MCP's total behavior
    s = pd.Series(task_scores)
    counts = s.value_counts()
    
    return pd.Series([
        s.mean(),
        counts.get(1, 0),
        counts.get(2, 0),
        counts.get(3, 0),
        counts.get(4, 0),
        counts.get(5, 0),
        len(s),
        has_high_physical
    ], index=['mean', 'count_1', 'count_2', 'count_3', 'count_4', 'count_5', 'total_ratings', 'is_high_phys_mcp'])

# 2. Apply the parsing logic to every MCP
mcp_stats = mcp_df.apply(parse_and_analyze, axis=1)

# 3. Split the groups
high_phys_group = mcp_stats[mcp_stats['is_high_phys_mcp'] == True]
others_group = mcp_stats[mcp_stats['is_high_phys_mcp'] == False]

# 4. Generate the comparison table
comparison = pd.DataFrame({
    "Metric": ["Average Mean Rating", "Avg Count of 1s", "Avg Count of 2s", "Avg Count of 3s", "Avg Count of 4s", "Avg Count of 5s", "Sample Size (MCPs)"],
    "High Physical Group (3-5 on Phys)": [
        high_phys_group['mean'].mean(),
        high_phys_group['count_1'].mean(),
        high_phys_group['count_2'].mean(),
        high_phys_group['count_3'].mean(),
        high_phys_group['count_4'].mean(),
        high_phys_group['count_5'].mean(),
        len(high_phys_group)
    ],
    "Other MCPs": [
        others_group['mean'].mean(),
        others_group['count_1'].mean(),
        others_group['count_2'].mean(),
        others_group['count_3'].mean(),
        others_group['count_4'].mean(),
        others_group['count_5'].mean(),
        len(others_group)
    ]
})

print("\n--- Comparison of Rating Behavior ---")
print(comparison.to_string(index=False))


--- Comparison of Rating Behavior ---
             Metric  High Physical Group (3-5 on Phys)  Other MCPs
Average Mean Rating                           2.179467    1.658266
    Avg Count of 1s                          35.088050   64.077277
    Avg Count of 2s                          40.553100   33.994527
    Avg Count of 3s                          31.184367   16.396891
    Avg Count of 4s                          11.511590    3.815236
    Avg Count of 5s                           0.964061    0.184982
 Sample Size (MCPs)                        5565.000000 4575.000000


In [2]:
import pandas as pd
import re
import numpy as np
from langdetect import detect, DetectorFactory

# Ensure reproducible language detection
DetectorFactory.seed = 0

def normalize(t):
    return re.sub(r"[^\w\s]", "", str(t).lower().strip())

def is_not_english(text):
    if not isinstance(text, str) or len(text.strip()) < 5:
        return False
    try:
        return detect(text) != 'en'
    except:
        return False

# 1. Load data
mcp_df = pd.read_csv("../data/mcp_results_2026-02-18.csv")
task_ref = pd.read_csv("../data/task_results_2026-02-18_with_physical.csv")

physical_tasks = set(task_ref[task_ref['physical'] == True]['task'].apply(normalize))

def parse_and_analyze(row):
    ratings_str = str(row['task_ratings'])
    parts = ratings_str.split(';')
    
    task_scores = []
    has_high_physical = False
    
    for part in parts:
        if ':' not in part: continue
        sub_parts = part.rsplit(':', 1)
        if len(sub_parts) < 2: continue
        
        raw_task_info = sub_parts[0].strip()
        try:
            score = int(re.search(r'\d+', sub_parts[1]).group())
        except: continue
        
        task_desc = re.sub(r'\s*\([^)]*\)$', '', raw_task_info).strip()
        if normalize(task_desc) in physical_tasks and score >= 3:
            has_high_physical = True
        
        task_scores.append(score)

    # Define the exact columns we want in our output
    cols = ['mean', 'count_1', 'count_2', 'count_3', 'count_4', 'count_5', 
            'text_len', 'is_non_english', 'is_high_phys_mcp', 'total_ratings']

    if not task_scores:
        return pd.Series([np.nan] * len(cols), index=cols)

    s = pd.Series(task_scores)
    counts = s.value_counts()
    
    text_content = str(row.get('text_for_llm', ""))
    title_content = str(row.get('title', ""))
    
    return pd.Series({
        'mean': s.mean(),
        'count_1': counts.get(1, 0),
        'count_2': counts.get(2, 0),
        'count_3': counts.get(3, 0),
        'count_4': counts.get(4, 0),
        'count_5': counts.get(5, 0),
        'text_len': len(text_content),
        'is_non_english': is_not_english(title_content),
        'is_high_phys_mcp': has_high_physical,
        'total_ratings': len(s)
    })

# 2. Process
mcp_stats = mcp_df.apply(parse_and_analyze, axis=1)

# 3. Grouping
high_phys = mcp_stats[mcp_stats['is_high_phys_mcp'] == True].copy()
others = mcp_stats[mcp_stats['is_high_phys_mcp'] == False].copy()

# 4. Final Comparison Table
def get_group_metrics(df):
    if df.empty:
        return {k: 0 for k in ["Avg Mean Rating", "Avg Count of 5s", "Avg Count of 1s", 
                               "Text Length (Mean)", "Text Length (Std)", "Text Length (Median)",
                               "Non-English Title Count", "Non-English Title %", "Sample Size"]}
    return {
        "Avg Mean Rating": df['mean'].mean(),
        "Avg Count of 5s": df['count_5'].mean(),
        "Avg Count of 1s": df['count_1'].mean(),
        "Text Length (Mean)": df['text_len'].mean(),
        "Text Length (Std)": df['text_len'].std(),
        "Text Length (Median)": df['text_len'].median(),
        "Non-English Title Count": df['is_non_english'].sum(),
        "Non-English Title %": f"{(df['is_non_english'].mean()*100):.1f}%",
        "Sample Size": len(df)
    }

metrics_high = get_group_metrics(high_phys)
metrics_others = get_group_metrics(others)

comparison = pd.DataFrame({
    "Metric": metrics_high.keys(),
    "High Physical Group": metrics_high.values(),
    "Other Group": metrics_others.values()
})

print("\n--- Final Analysis: Physical Task Bias & Metadata ---")
print(comparison.to_string(index=False))


--- Final Analysis: Physical Task Bias & Metadata ---
                 Metric High Physical Group Other Group
        Avg Mean Rating            2.179467    1.658266
        Avg Count of 5s            0.964061    0.184982
        Avg Count of 1s            35.08805   64.077277
     Text Length (Mean)          516.484277  517.906743
      Text Length (Std)          295.196632  171.642063
   Text Length (Median)               488.0       493.0
Non-English Title Count                2365        1981
    Non-English Title %               42.5%       43.4%
            Sample Size                5565        4568


## What is the makeup of physical true or false tasks in the aei data and my mcp data

In [ ]:
#Helper Functions

def normalize_text(text):
    if not isinstance(text, str):
        return text
    text = text.lower().strip()                   # lowercase + trim
    text = re.sub(r"[^\w\s]", "", text)            # remove punctuation
    text = re.sub(r"\s+", " ", text)               # collapse multiple spaces
    return text



# Load mapping and normalize the join key
physical_source = pd.read_csv("../data/tasks_dwa_iwa_gwa_v30.1_physical.csv")
physical_mapping = physical_source[['task', 'physical']].drop_duplicates('task').copy()
physical_mapping['task_norm'] = physical_mapping['task'].apply(normalize_text)

# Define files and their specific task column names
files_to_check = {
    "v1": {"path": "../data/task_pct_v1.csv", "key": "task_name"},
    "v2": {"path": "../data/task_pct_v2.csv", "key": "task_name"},
    "v3": {"path": "../data/task_pct_v3.csv", "key": "task_name"},
    "results_02_18": {"path": "../data/task_results_2026-02-18.csv", "key": "task"}
}

results = []

for name, info in files_to_check.items():
    target_df = pd.read_csv(info["path"])
    
    # Create normalized temporary key for merging
    target_df['temp_norm_key'] = target_df[info["key"]].apply(normalize_text)
    
    # Merge using normalized keys
    merged = pd.merge(
        target_df, 
        physical_mapping[['task_norm', 'physical']], 
        left_on='temp_norm_key', 
        right_on='task_norm', 
        how='left'
    )
    
    # Clean up temporary/redundant columns
    merged = merged.drop(columns=['temp_norm_key', 'task_norm'])
    
    # Calculate counts
    total = len(merged)
    trues = (merged['physical'] == True).sum()
    falses = (merged['physical'] == False).sum()
    missing = merged['physical'].isna().sum()

    # Calculate Stats with Percentages
    stats = {
        "File": name,
        "Total Rows": total,
        "True": trues,
        "True %": f"{(trues/total)*100:.1f}%" if total > 0 else "0%",
        "False": falses,
        "False %": f"{(falses/total)*100:.1f}%" if total > 0 else "0%",
        "Missing": missing,
        "Missing %": f"{(missing/total)*100:.1f}%" if total > 0 else "0%"
    }
    results.append(stats)
    
    # Save output
    merged.to_csv(info["path"].replace(".csv", "_with_physical.csv"), index=False)

# Output Summary Table
summary_df = pd.DataFrame(results)
print("\n--- Physical Task Merge Summary ---")
print(summary_df.to_string(index=False))

# Aggregated totals across all files
print("\n--- Aggregated Totals ---")
agg_total = summary_df['Total Rows'].sum()
agg_missing = summary_df['Missing'].sum()
print(f"Total Tasks Processed: {agg_total}")
print(f"Total Missing Across All: {agg_missing}")
print(f"Overall Coverage: {((agg_total - agg_missing) / agg_total * 100):.2f}%")


--- Physical Task Merge Summary ---
         File  Total Rows  True True %  False False %  Missing Missing %
           v1        3530   489  13.9%   2631   74.5%      410     11.6%
           v2        3380   458  13.6%   2527   74.8%      395     11.7%
           v3        2629   371  14.1%   1942   73.9%      316     12.0%
results_02_18       11653  3649  31.3%   8002   68.7%        2      0.0%

--- Aggregated Totals ---
Total Tasks Processed: 21192
Total Missing Across All: 1123
Overall Coverage: 94.70%


## Makeup of dates in MCP data

In [2]:
import pandas as pd

# Load file
df = pd.read_csv('../data/mcp_results_2026-02-18.csv')

# Unique values
print("Unique uploaded_clean values:")
print(df['uploaded_clean'].unique())

# Counts per value
print("\nCounts of each uploaded_clean value:")
print(df['uploaded_clean'].value_counts(dropna=False))

# Optional: percentages too
print("\nPercent distribution:")
print(df['uploaded_clean'].value_counts(dropna=False, normalize=True) * 100)

Unique uploaded_clean values:
['2025-05-19' '2025-07-18' nan '2025-06-18' '2025-08-17' '2025-09-16'
 '2025-10-16' '2025-04-19' '2025-11-15' '2026-01-29' '2025-12-15'
 '2026-01-31' '2026-02-01' '2026-01-25' '2026-02-06' '2026-01-23'
 '2026-01-24' '2026-01-28' '2026-01-26' '2026-02-11' '2026-02-03'
 '2026-02-07' '2026-02-10' '2026-02-08' '2026-01-22' '2025-06-23'
 '2025-07-23' '2025-04-24' '2025-05-24' '2026-02-09' '2026-02-16'
 '2026-02-14' '2025-09-21' '2025-10-21' '2025-11-20' '2026-02-02'
 '2025-12-20' '2025-08-22' '2026-02-15' '2026-02-04' '2026-02-12'
 '2026-02-05' '2026-01-30' '2026-02-13']

Counts of each uploaded_clean value:
uploaded_clean
2025-05-19    3240
2025-04-19    2344
2025-06-18    1519
2025-07-18     934
NaN            899
2025-04-24     634
2025-05-24     143
2025-08-17      70
2025-06-23      69
2025-09-16      68
2025-11-15      48
2025-12-15      45
2025-10-16      38
2026-02-07       7
2025-12-20       6
2026-02-10       6
2025-10-21       5
2026-01-29       5
20

## How many tasks match the most current task statements v30.1 in all of our datasets

In [2]:
import pandas as pd
from pathlib import Path

# --- FILE PATHS ---
base_dir = "../data/" # adjust if needed
files = [
    "first_pass_aei_api_v3.csv",
    "first_pass_aei_api_v4.csv",
    "first_pass_aei_v1.csv",
    "first_pass_aei_v2.csv",
    "first_pass_aei_v3.csv",
    "first_pass_aei_v4.csv",
    "first_pass_mcp_v4.csv",
    "first_pass_microsoft.csv",
]

statements_path = Path("../data/task_statements_v30.1.csv")

# --- LOAD STATEMENTS ---
statements = pd.read_csv(statements_path)

# Apply normalize() to Task column
statements["task_normalized"] = statements["Task"].apply(normalize_text)


# UNIQUE statement tasks
statement_set = set(statements["task_normalized"].dropna().unique())

results = []

for f in files:
    df = pd.read_csv(f"{base_dir}{f}")

    # UNIQUE dataset tasks
    df_unique = df.dropna(subset=["task_normalized"]).drop_duplicates(subset=["task_normalized"])

    total_tasks = len(df_unique)

    matched_mask = df_unique["task_normalized"].isin(statement_set)

    n_matched = matched_mask.sum()
    n_unmatched = total_tasks - n_matched

    pct_matched = n_matched / total_tasks * 100
    pct_unmatched = n_unmatched / total_tasks * 100

    unmatched_df = df_unique.loc[~matched_mask]

    unmatched_pct_sum = unmatched_df["pct_normalized"].sum()
    unmatched_pct_mean = unmatched_df["pct_normalized"].mean()

    results.append({
        "dataset": f,
        "total_unique_tasks": total_tasks,
        "matched_unique_n": n_matched,
        "matched_unique_%": pct_matched,
        "unmatched_unique_n": n_unmatched,
        "unmatched_unique_%": pct_unmatched,
        "unmatched_pct_normalized_sum": unmatched_pct_sum,
        "unmatched_pct_normalized_mean": unmatched_pct_mean
    })

summary_df = pd.DataFrame(results).sort_values("dataset")
summary_df

,dataset,total_unique_tasks,matched_unique_n,matched_unique_%,unmatched_unique_n,unmatched_unique_%,unmatched_pct_normalized_sum,unmatched_pct_normalized_mean
0,first_pass_aei_api_v3.csv,2051,1812,88.347148,239,11.652852,15.371122,0.064314
1,first_pass_aei_api_v4.csv,2248,1995,88.745552,253,11.254448,15.747394,0.062243
2,first_pass_aei_v1.csv,3509,3100,88.344258,409,11.655742,16.111427,0.039392
3,first_pass_aei_v2.csv,3360,2966,88.273810,394,11.726190,15.477765,0.039284
4,first_pass_aei_v3.csv,2612,2298,87.978560,314,12.021440,13.717079,0.043685
5,first_pass_aei_v4.csv,3163,2793,88.302245,370,11.697755,13.059712,0.035297
6,first_pass_mcp_v4.csv,8905,8904,99.988770,1,0.011230,0.000004,0.000004
7,first_pass_microsoft.csv,8319,8319,100.000000,0,0.000000,0.000000,NaN


In [3]:
import pandas as pd
from pathlib import Path

# Paths to your files (adjust if they are in different subfolders)
CUMULATIVE_DIR = Path("../data/final/cumulative")
FINAL_DIR = Path("../data/final")

# 1. Load the files
aei_v4 = pd.read_csv(CUMULATIVE_DIR / "final_aei_cumulative_v4.csv")
eco_2025 = pd.read_csv(FINAL_DIR / "final_eco_2025.csv")

# 2. Get unique tasks from each
aei_tasks = set(aei_v4["task_normalized"].unique())
eco_tasks = set(eco_2025["task_normalized"].unique())

# 3. Calculate intersections
matched_tasks = aei_tasks.intersection(eco_tasks)
missing_in_eco = aei_tasks - eco_tasks

# 4. Print results
print(f"--- Task Matching Analysis ---")
print(f"Total Unique Tasks in AEI v4: {len(aei_tasks)}")
print(f"Total Unique Tasks in Eco 2025: {len(eco_tasks)}")
print(f"-" * 30)
print(f"Successfully Matched: {len(matched_tasks)}")
print(f"Tasks in AEI but NOT in Eco: {len(missing_in_eco)}")
print(f"Match Rate: {(len(matched_tasks) / len(aei_tasks)):.1%}")

# Optional: See a few examples of tasks that didn't match
if missing_in_eco:
    print(f"\nExample of missing tasks (first 5):")
    print(list(missing_in_eco)[:5])

--- Task Matching Analysis ---
Total Unique Tasks in AEI v4: 4589
Total Unique Tasks in Eco 2025: 17507
------------------------------
Successfully Matched: 4063
Tasks in AEI but NOT in Eco: 526
Match Rate: 88.5%

Example of missing tasks (first 5):
['coordinate due diligence processes and the negotiation or execution of purchase or sale agreements', 'select plays or scripts for production and determine how material should be interpreted and performed', 'prepare reports to communicate information such as contamination test results decontamination results or decontamination procedures', 'use computers and synthesizers to compose orchestrate and arrange music', 'develop instructional materials and products for technologybased redesign of courses']


## AEI data can't have taxonomy for each task as some are missing from the tasks to dwa file

In [5]:
import pandas as pd
import re
from pathlib import Path

# Assuming the same directory structure from before
ONET_DIR = Path("../data/onet")  # Adjust if the files are elsewhere

def normalize_text(text):
    if not isinstance(text, str):
        return text
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

# Load files
# (Adding .csv to the one that was missing it)
df_statements = pd.read_csv("../data/task_statements_v20.1.csv")
df_dwas = pd.read_csv("../data/tasks_to_dwas_v20.1.csv")

# Apply normalization
# Note: Check if the column is exactly 'Task' or something like 'Task Statement'
# O*NET files often use 'Task' or 'Task Statement'. 
statements_tasks = set(df_statements['Task'].apply(normalize_text))
dwas_tasks = set(df_dwas['Task'].apply(normalize_text))

# Comparison
overlap = statements_tasks.intersection(dwas_tasks)
only_statements = statements_tasks - dwas_tasks
only_dwas = dwas_tasks - statements_tasks

print(f"Total tasks in Statements: {len(statements_tasks)}")
print(f"Total tasks in DWAs:       {len(dwas_tasks)}")
print("-" * 30)
print(f"Tasks in both:             {len(overlap)}")
print(f"Tasks only in Statements:  {len(only_statements)}")
print(f"Tasks only in DWAs:        {len(only_dwas)}")

Total tasks in Statements: 18405
Total tasks in DWAs:       17254
------------------------------
Tasks in both:             17254
Tasks only in Statements:  1151
Tasks only in DWAs:        0


In [8]:
# 1. Identify the unmapped tasks (using the normalized versions)
df_statements['normalized_task'] = df_statements['Task'].apply(normalize_text)
orphans_df = df_statements[df_statements['normalized_task'].isin(only_statements)]

# 2. Check Task Type distribution (Core vs Supplemental)
# O*NET uses 'Task Type' column in the Statements file
type_pattern = orphans_df['Task Type'].value_counts(normalize=True) * 100

# 3. See which Occupations (SOC codes) have the most unmapped tasks
top_orphan_occupations = orphans_df.groupby(['O*NET-SOC Code', 'Title']).size().sort_values(ascending=False).head(10)

print("--- PATTERN ANALYSIS ---")
print(f"\nTask Type Breakdown for Unmapped Tasks:")
print(type_pattern)

print(f"\nTop 10 Occupations with Unmapped Tasks:")
print(top_orphan_occupations)

# 4. Optional: Look at a few actual task strings to see if they look "niche"
print(f"\nSample of Unmapped Task Statements:")
print(orphans_df['Task'].sample(min(5, len(orphans_df))).values)

--- PATTERN ANALYSIS ---

Task Type Breakdown for Unmapped Tasks:
Task Type
Core            54.770318
Supplemental    45.229682
Name: proportion, dtype: float64

Top 10 Occupations with Unmapped Tasks:
O*NET-SOC Code  Title                                      
53-6041.00      Traffic Technicians                            10
41-2011.00      Cashiers                                        8
19-4099.01      Quality Control Analysts                        8
43-4121.00      Library Assistants, Clerical                    7
51-9198.00      Helpers--Production Workers                     7
21-2011.00      Clergy                                          7
23-2093.00      Title Examiners, Abstractors, and Searchers     7
37-2012.00      Maids and Housekeeping Cleaners                 7
19-1031.03      Park Naturalists                                7
43-4031.02      Municipal Clerks                                6
dtype: int64

Sample of Unmapped Task Statements:
['Confer with park staff to 

## Rating distribution in MCP data to get imputing distribution for microsoft auto/aug scale

In [2]:
import pandas as pd
import re
from pathlib import Path
from collections import Counter

data_dir = Path("../data")

df = pd.read_csv(data_dir / "mcp_results_2026-02-18.csv")

rating_counts = Counter()

for cell in df["task_rating_response_raw"].dropna():

    # remove xml wrapper
    cell = cell.replace("<ratings>", "").replace("</ratings>", "").strip()

    # find all ratings after colon
    ratings = re.findall(r":(\d+)", cell)

    rating_counts.update(map(int, ratings))

total = sum(rating_counts.values())

print("\nCounts and Percentages:")
for r in range(1, 6):
    count = rating_counts.get(r, 0)
    pct = (count / total) * 100 if total > 0 else 0
    print(f"Rating {r}: {count:,}  ({pct:.2f}%)")

print(f"\nTotal ratings: {total:,}")


Counts and Percentages:
Rating 1: 487,970  (40.49%)
Rating 2: 380,965  (31.61%)
Rating 3: 248,442  (20.62%)
Rating 4: 81,490  (6.76%)
Rating 5: 6,210  (0.52%)

Total ratings: 1,205,077


## MCP Conditional Distributions for Microsoft Auto/Aug Imputing

In [1]:
"""
MCP Conditional Distribution Table — for report/notebook display.
Shows how rating distributions shift as % moderate+ increases.
"""

import pandas as pd
import numpy as np
import re
from pathlib import Path
from collections import Counter

data_dir = Path("../data")
mcp_df = pd.read_csv(data_dir / "mcp_results_2026-02-18.csv")

# Parse MCP rating distributions
mcp_dists = []
for idx, cell in mcp_df["task_rating_response_raw"].dropna().items():
    cell = cell.replace("<ratings>", "").replace("</ratings>", "").strip()
    ratings = list(map(int, re.findall(r":(\d+)", cell)))
    if len(ratings) == 0:
        continue
    counts = Counter(ratings)
    n = len(ratings)
    n_mod_plus = counts.get(3, 0) + counts.get(4, 0) + counts.get(5, 0)
    mcp_dists.append({
        'n_ratings': n,
        'pct_mod_plus': n_mod_plus / n if n > 0 else 0,
        'count_1': counts.get(1, 0),
        'count_2': counts.get(2, 0),
        'count_3': counts.get(3, 0),
        'count_4': counts.get(4, 0),
        'count_5': counts.get(5, 0),
    })

mcp_dist_df = pd.DataFrame(mcp_dists)

# Bin by % moderate+
bins = np.arange(0, 1.05, 0.05)
bin_labels = [f"{b:.0%}-{b+0.05:.0%}" for b in bins[:-1]]
mcp_dist_df['scope_bin'] = pd.cut(mcp_dist_df['pct_mod_plus'], bins=bins, 
                                   labels=bin_labels, include_lowest=True)

# Build table
rows = []
for bin_label, group in mcp_dist_df.groupby('scope_bin', observed=True):
    totals = group[['count_1', 'count_2', 'count_3', 'count_4', 'count_5']].sum()
    total = totals.sum()
    if total > 0:
        mean = sum((i+1) * totals[f'count_{i+1}'] for i in range(5)) / total
        rows.append({
            'Scope Bin (% Moderate+)': bin_label,
            'N MCPs': len(group),
            'N Ratings': int(total),
            '% Rating 1': f"{totals['count_1']/total:.1%}",
            '% Rating 2': f"{totals['count_2']/total:.1%}",
            '% Rating 3': f"{totals['count_3']/total:.1%}",
            '% Rating 4': f"{totals['count_4']/total:.1%}",
            '% Rating 5': f"{totals['count_5']/total:.1%}",
            'Mean': round(mean, 2),
        })

table_df = pd.DataFrame(rows)
print(table_df.to_string(index=False))

# Also show overall stats
print(f"\nOverall: {len(mcp_dist_df)} MCPs, {mcp_dist_df['n_ratings'].sum()} total ratings")
print(f"Mean pct_mod_plus: {mcp_dist_df['pct_mod_plus'].mean():.3f}")
print(f"Median pct_mod_plus: {mcp_dist_df['pct_mod_plus'].median():.3f}")

# Global distribution (all ratings pooled)
global_totals = mcp_dist_df[['count_1','count_2','count_3','count_4','count_5']].sum()
global_total = global_totals.sum()
print(f"\nGlobal distribution:")
for i in range(1, 6):
    print(f"  Rating {i}: {global_totals[f'count_{i}']:,.0f} ({global_totals[f'count_{i}']/global_total:.1%})")
global_mean = sum((i+1) * global_totals[f'count_{i+1}'] for i in range(5)) / global_total
print(f"  Global mean: {global_mean:.3f}")

# Distribution excluding 1s (sensitivity check)
global_no1 = global_total - global_totals['count_1']
print(f"\nGlobal distribution excluding 1s:")
for i in range(2, 6):
    print(f"  Rating {i}: {global_totals[f'count_{i}']:,.0f} ({global_totals[f'count_{i}']/global_no1:.1%})")
global_mean_no1 = sum((i+1) * global_totals[f'count_{i+1}'] for i in range(1, 5)) / global_no1
print(f"  Mean excluding 1s: {global_mean_no1:.3f}")

Scope Bin (% Moderate+)  N MCPs  N Ratings % Rating 1 % Rating 2 % Rating 3 % Rating 4 % Rating 5  Mean
                  0%-5%    1765     203298      81.3%      17.3%       1.2%       0.2%       0.0%  1.20
                 5%-10%    1005     119433      67.6%      24.9%       6.0%       1.3%       0.1%  1.41
                10%-15%     869     102873      56.2%      31.1%      10.2%       2.3%       0.2%  1.59
                15%-20%     856     102698      47.0%      35.5%      14.0%       3.3%       0.2%  1.74
                20%-25%     739      88462      38.1%      39.3%      18.2%       4.2%       0.2%  1.89
                25%-30%     740      88249      32.4%      40.0%      21.3%       6.0%       0.4%  2.02
                30%-35%     679      81625      27.0%      40.5%      25.4%       6.6%       0.4%  2.13
                35%-40%     611      73155      21.9%      40.6%      28.7%       8.3%       0.5%  2.25
                40%-45%     587      70532      17.4%      40.2%

## Aei vs Modern Task List

In [17]:
import pandas as pd
import os
from pathlib import Path

DATA_DIR = Path("../data")



def run_alignment_analysis():
    eco_25 = pd.read_csv(DATA_DIR / "second_pass_eco_2025.csv")
    
    aei_files = sorted([f for f in os.listdir(DATA_DIR) if f.startswith("second_pass_aei")])
    if not aei_files:
        print("No AEI second pass files found.")
        return

    # Deduplicate eco to unique (title_current, task_normalized) pairs
    eco_pairs = eco_25[["title_current", "task_normalized"]].drop_duplicates()
    eco_titles = set(eco_pairs["title_current"].unique())
    eco_tasks_all = set(eco_pairs["task_normalized"].unique())

    print(f"=== BASELINE: second_pass_eco_2025 ===")
    print(f"Unique titles: {len(eco_titles)}")
    print(f"Unique tasks:  {len(eco_tasks_all)}")
    print(f"Unique (title, task) pairs: {len(eco_pairs)}")

    all_mismatches = []

    aei_all = pd.concat(
        [pd.read_csv(DATA_DIR / f) for f in aei_files],
        ignore_index=True
    )
    aei_pairs = aei_all[["title", "task_normalized"]].drop_duplicates()
    v_name = "aei_all"

    # Deduplicate AEI to unique (title, task_normalized) pairs
    # AEI uses "title" where eco uses "title_current"

    aei_titles = set(aei_pairs["title"].unique())
    aei_tasks_all = set(aei_pairs["task_normalized"].unique())

    print(f"\n{'='*60}")
    print(f"=== {v_name} ===")
    print(f"Unique titles: {len(aei_titles)}")
    print(f"Unique tasks:  {len(aei_tasks_all)}")
    print(f"Unique (title, task) pairs: {len(aei_pairs)}")

    # --- TITLE-LEVEL ALIGNMENT ---
    matched_titles = eco_titles & aei_titles
    only_eco_titles = eco_titles - aei_titles
    only_aei_titles = aei_titles - eco_titles

    print(f"\n--- Title Alignment ---")
    print(f"Matched titles:    {len(matched_titles)} / {len(eco_titles)} eco ({len(matched_titles)/len(eco_titles):.1%}), "
            f"{len(matched_titles)} / {len(aei_titles)} aei ({len(matched_titles)/len(aei_titles):.1%})")
    print(f"Only in Eco 2025:  {len(only_eco_titles)}")
    print(f"Only in AEI:       {len(only_aei_titles)}")

    # --- TASK ALIGNMENT WITHIN MATCHED TITLES ---
    eco_matched = eco_pairs[eco_pairs["title_current"].isin(matched_titles)]
    aei_matched = aei_pairs[aei_pairs["title"].isin(matched_titles)]

    # Convert to sets of (title, task) tuples for comparison
    eco_set = set(zip(eco_matched["title_current"], eco_matched["task_normalized"]))
    aei_set = set(zip(aei_matched["title"], aei_matched["task_normalized"]))

    shared_pairs = eco_set & aei_set
    only_eco_pairs = eco_set - aei_set
    only_aei_pairs = aei_set - eco_set

    print(f"\n--- Task Alignment (within matched titles) ---")
    print(f"Shared (title, task) pairs: {len(shared_pairs)} / {len(eco_set)} eco ({len(shared_pairs)/len(eco_set):.1%})")
    print(f"Only in Eco 2025:           {len(only_eco_pairs)}")
    print(f"Only in AEI:                {len(only_aei_pairs)}")

    # Per-title breakdown stats
    title_stats = []
    for title in matched_titles:
        eco_t = set(eco_matched[eco_matched["title_current"] == title]["task_normalized"])
        aei_t = set(aei_matched[aei_matched["title"] == title]["task_normalized"])
        shared = eco_t & aei_t
        title_stats.append({
            "title": title,
            "eco_tasks": len(eco_t),
            "aei_tasks": len(aei_t),
            "shared": len(shared),
            "only_eco": len(eco_t - aei_t),
            "only_aei": len(aei_t - eco_t),
            "match_rate": len(shared) / len(eco_t) if eco_t else 0,
        })
    title_stats_df = pd.DataFrame(title_stats).sort_values("match_rate")

    print(f"\nPer-title match rate: mean={title_stats_df['match_rate'].mean():.1%}, "
            f"median={title_stats_df['match_rate'].median():.1%}")
    print(f"Titles with 100% match: {(title_stats_df['match_rate'] == 1.0).sum()}")
    print(f"Titles with <50% match: {(title_stats_df['match_rate'] < 0.5).sum()}")

    print(f"\nWorst 5 title matches:")
    print(title_stats_df[["title", "eco_tasks", "aei_tasks", "shared", "match_rate"]].head(5).to_string(index=False))

    # --- TASK ALIGNMENT ACROSS UNMATCHED TITLES ---
    # For titles only in eco, how many of their tasks exist somewhere in AEI?
    eco_unmatched_tasks = set(
        eco_pairs[eco_pairs["title_current"].isin(only_eco_titles)]["task_normalized"]
    )
    eco_unmatched_in_aei = eco_unmatched_tasks & aei_tasks_all

    print(f"\n--- Tasks from unmatched eco titles ---")
    print(f"Unique tasks in eco-only titles: {len(eco_unmatched_tasks)}")
    print(f"Of those, found elsewhere in AEI: {len(eco_unmatched_in_aei)} ({len(eco_unmatched_in_aei)/len(eco_unmatched_tasks):.1%})" if eco_unmatched_tasks else "N/A")

    # Same for AEI-only titles
    aei_unmatched_tasks = set(
        aei_pairs[aei_pairs["title"].isin(only_aei_titles)]["task_normalized"]
    )
    aei_unmatched_in_eco = aei_unmatched_tasks & eco_tasks_all

    print(f"Unique tasks in aei-only titles: {len(aei_unmatched_tasks)}")
    print(f"Of those, found elsewhere in Eco: {len(aei_unmatched_in_eco)} ({len(aei_unmatched_in_eco)/len(aei_unmatched_tasks):.1%})" if aei_unmatched_tasks else "N/A")

    # --- GLOBAL TASK OVERLAP (ignoring titles entirely) ---
    task_overlap = eco_tasks_all & aei_tasks_all
    print(f"\n--- Global task overlap (title-agnostic) ---")
    print(f"Shared tasks: {len(task_overlap)} / {len(eco_tasks_all)} eco ({len(task_overlap)/len(eco_tasks_all):.1%}), "
            f"{len(task_overlap)} / {len(aei_tasks_all)} aei ({len(task_overlap)/len(aei_tasks_all):.1%})")

    # Collect mismatches
    for title, task in only_eco_pairs:
        all_mismatches.append({"version": v_name, "title": title, "task": task, "status": "Only in Eco 2025"})
    for title, task in only_aei_pairs:
        all_mismatches.append({"version": v_name, "title": title, "task": task, "status": "Only in AEI"})

    # Save
    if all_mismatches:
        mismatch_df = pd.DataFrame(all_mismatches)
        mismatch_df.to_csv(DATA_DIR / "task_drift_mismatch_report.csv", index=False)
        print(f"\nSaved mismatch report: {len(mismatch_df)} rows")

run_alignment_analysis()

=== BASELINE: second_pass_eco_2025 ===
Unique titles: 923
Unique tasks:  17507
Unique (title, task) pairs: 18796

=== aei_all ===
Unique titles: 826
Unique tasks:  4881
Unique (title, task) pairs: 5730

--- Title Alignment ---
Matched titles:    633 / 923 eco (68.6%), 633 / 826 aei (76.6%)
Only in Eco 2025:  290
Only in AEI:       193

--- Task Alignment (within matched titles) ---
Shared (title, task) pairs: 4234 / 13105 eco (32.3%)
Only in Eco 2025:           8871
Only in AEI:                265

Per-title match rate: mean=31.9%, median=27.3%
Titles with 100% match: 0
Titles with <50% match: 486

Worst 5 title matches:
                                                 title  eco_tasks  aei_tasks  shared  match_rate
                                      Floral Designers         15          1       0         0.0
                         Medical Appliance Technicians         15          1       0         0.0
                                   Tool and Die Makers         17          1    

In [18]:
import pandas as pd
import os
from pathlib import Path

DATA_DIR = Path("../data")

def normalize_text(text):
    if not isinstance(text, str):
        return ""
    return text.lower().strip().replace(",", "").replace(".", "").replace("'", "").replace('"', '')

# ============================================================
# 1. Build master AEI (title, task_normalized) deduplicated
# ============================================================
aei_files = sorted([f for f in os.listdir(DATA_DIR) if f.startswith("second_pass_aei")])
aei_all = pd.concat([pd.read_csv(DATA_DIR / f) for f in aei_files], ignore_index=True)
aei_pairs = aei_all[["title", "task_normalized"]].drop_duplicates()
print(f"Master AEI: {len(aei_pairs)} unique (title, task) pairs")
print(f"  Unique titles: {aei_pairs['title'].nunique()}")
print(f"  Unique tasks:  {aei_pairs['task_normalized'].nunique()}")

# ============================================================
# 2. Load and prep 2025 task ratings
# ============================================================
ratings_raw = pd.read_csv(DATA_DIR / "task_ratings_v30.1.csv")
ratings_raw["task_normalized"] = ratings_raw["Task"].apply(normalize_text)
ratings_raw["title_normalized"] = ratings_raw["Title"].apply(normalize_text)

# Pivot to one row per (title, task) with freq_mean, importance, relevance
freq = ratings_raw[ratings_raw["Scale ID"] == "FT"].copy()
freq_mean = freq.groupby(["title_normalized", "task_normalized"]).apply(
    lambda g: (g["Category"].astype(float) * g["Data Value"] / 100).sum()
).reset_index(name="freq_mean")

imp = (
    ratings_raw[ratings_raw["Scale ID"] == "IM"]
    [["title_normalized", "task_normalized", "Data Value"]]
    .rename(columns={"Data Value": "importance"})
)

rel = (
    ratings_raw[ratings_raw["Scale ID"] == "RT"]
    [["title_normalized", "task_normalized", "Data Value"]]
    .rename(columns={"Data Value": "relevance"})
)

ratings_clean = freq_mean.merge(imp, on=["title_normalized", "task_normalized"], how="outer")
ratings_clean = ratings_clean.merge(rel, on=["title_normalized", "task_normalized"], how="outer")

print(f"\n2025 Ratings: {len(ratings_clean)} unique (title, task) pairs")

# ============================================================
# 3. Normalize AEI title for matching
# ============================================================
aei_pairs["title_normalized"] = aei_pairs["title"].apply(normalize_text)

# ============================================================
# 4a. Merge on (title_normalized, task_normalized)
# ============================================================
merged_both = aei_pairs.merge(
    ratings_clean,
    on=["title_normalized", "task_normalized"],
    how="left",
    indicator=True
)
matched_both = (merged_both["_merge"] == "both").sum()
total = len(merged_both)
print(f"\n--- Merge on (title + task) ---")
print(f"Matched: {matched_both} / {total} ({matched_both/total:.1%})")
print(f"Missing: {total - matched_both}")

# ============================================================
# 4b. For unmatched, try task-only merge (avg ratings across occupations)
# ============================================================
unmatched = merged_both[merged_both["_merge"] == "left_only"].copy()
unmatched.drop(columns=["freq_mean", "importance", "relevance", "_merge"], inplace=True)

ratings_task_only = (
    ratings_clean
    .groupby("task_normalized")[["freq_mean", "importance", "relevance"]]
    .mean()
    .reset_index()
)

merged_task_only = unmatched.merge(
    ratings_task_only,
    on="task_normalized",
    how="left",
    indicator=True
)
matched_task = (merged_task_only["_merge"] == "both").sum()
print(f"\n--- Merge on (task only, avg across occs) ---")
print(f"Matched: {matched_task} / {len(unmatched)} ({matched_task/len(unmatched):.1%})")
print(f"Still missing: {len(unmatched) - matched_task}")

# ============================================================
# 5. Combine and summarize
# ============================================================
matched_df = merged_both[merged_both["_merge"] == "both"].drop(columns=["_merge"])
task_only_df = merged_task_only.drop(columns=["_merge"])

final = pd.concat([matched_df, task_only_df], ignore_index=True)

total_with_ratings = final["freq_mean"].notna().sum()
total_missing = final["freq_mean"].isna().sum()
print(f"\n--- Final ---")
print(f"Total AEI pairs:    {len(final)}")
print(f"With 2025 ratings:  {total_with_ratings} ({total_with_ratings/len(final):.1%})")
print(f"Missing ratings:    {total_missing} ({total_missing/len(final):.1%})")

# Show some examples of what's missing
if total_missing > 0:
    missing = final[final["freq_mean"].isna()][["title", "task_normalized"]].head(10)
    print(f"\nExample missing pairs:")
    print(missing.to_string(index=False))

Master AEI: 5730 unique (title, task) pairs
  Unique titles: 826
  Unique tasks:  4881

2025 Ratings: 17951 unique (title, task) pairs

--- Merge on (title + task) ---
Matched: 3996 / 5730 (69.7%)
Missing: 1734

--- Merge on (task only, avg across occs) ---
Matched: 749 / 1734 (43.2%)
Still missing: 985

--- Final ---
Total AEI pairs:    5730
With 2025 ratings:  4745 (82.8%)
Missing ratings:    985 (17.2%)

Example missing pairs:
                              title                                                                                                                                       task_normalized
    General and Operations Managers                                                  develop or implement productmarketing strategies including advertising campaigns or sales promotions
Advertising and Promotions Managers               track program budgets and expenses and campaign response rates to evaluate each campaign based on program objectives and industry norms
Advertis

C:\Users\teddy\AppData\Local\Temp\ipykernel_7904\172964112.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  freq_mean = freq.groupby(["title_normalized", "task_normalized"]).apply(


In [19]:
# ============================================================
# Merge task ratings into eco 2015 and eco 2025
# ============================================================

for eco_file, title_col in [
    ("second_pass_eco_2025.csv", "title_current"),
    ("second_pass_eco_2015.csv", "title"),  # adjust if different col name
]:
    print(f"\n{'='*60}")
    print(f"=== {eco_file} ===")
    
    eco = pd.read_csv(DATA_DIR / eco_file)
    eco_pairs = eco[[title_col, "task_normalized"]].drop_duplicates()
    eco_pairs["title_normalized"] = eco_pairs[title_col].apply(normalize_text)
    
    print(f"Unique (title, task) pairs: {len(eco_pairs)}")
    print(f"Unique titles: {eco_pairs[title_col].nunique()}")
    print(f"Unique tasks:  {eco_pairs['task_normalized'].nunique()}")

    # 4a. Merge on (title_normalized, task_normalized)
    merged_both = eco_pairs.merge(
        ratings_clean,
        on=["title_normalized", "task_normalized"],
        how="left",
        indicator=True
    )
    matched_both = (merged_both["_merge"] == "both").sum()
    total = len(merged_both)
    print(f"\n--- Merge on (title + task) ---")
    print(f"Matched: {matched_both} / {total} ({matched_both/total:.1%})")
    print(f"Missing: {total - matched_both}")

    # 4b. Task-only fallback
    unmatched = merged_both[merged_both["_merge"] == "left_only"].copy()
    unmatched.drop(columns=["freq_mean", "importance", "relevance", "_merge"], inplace=True)

    merged_task_only = unmatched.merge(
        ratings_task_only,
        on="task_normalized",
        how="left",
        indicator=True
    )
    matched_task = (merged_task_only["_merge"] == "both").sum()
    print(f"\n--- Merge on (task only, avg across occs) ---")
    print(f"Matched: {matched_task} / {len(unmatched)} ({matched_task/len(unmatched):.1%})")
    print(f"Still missing: {len(unmatched) - matched_task}")

    # Combine
    matched_df = merged_both[merged_both["_merge"] == "both"].drop(columns=["_merge"])
    task_only_df = merged_task_only.drop(columns=["_merge"])
    final = pd.concat([matched_df, task_only_df], ignore_index=True)

    total_with = final["freq_mean"].notna().sum()
    total_miss = final["freq_mean"].isna().sum()
    print(f"\n--- Final ---")
    print(f"Total pairs:       {len(final)}")
    print(f"With 2025 ratings: {total_with} ({total_with/len(final):.1%})")
    print(f"Missing ratings:   {total_miss} ({total_miss/len(final):.1%})")

    if total_miss > 0:
        missing = final[final["freq_mean"].isna()][[title_col, "task_normalized"]].head(10)
        print(f"\nExample missing pairs:")
        print(missing.to_string(index=False))


=== second_pass_eco_2025.csv ===
Unique (title, task) pairs: 18796
Unique titles: 923
Unique tasks:  17507

--- Merge on (title + task) ---
Matched: 16609 / 18796 (88.4%)
Missing: 2187

--- Merge on (task only, avg across occs) ---
Matched: 113 / 2187 (5.2%)
Still missing: 2074

--- Final ---
Total pairs:       18796
With 2025 ratings: 16722 (89.0%)
Missing ratings:   2074 (11.0%)

Example missing pairs:
                  title_current                                                                                                                                                                                          task_normalized
               Chief Executives direct human resources activities including the approval of human resource plans or activities the selection of directors or other highlevel staff or establishment or organization of major departments
               Chief Executives                                                                                              

## Freq, Imp, Rel Correlations and Analysis (To see how task value is split up)

In [8]:
"""
Task Rating Correlation Analysis
Goal: Understand relationships between freq_mean, importance, and relevance
to inform how to split occupation-level wages across tasks.
"""

import pandas as pd
import numpy as np
from scipy import stats

# ── Load and dedup ──────────────────────────────────────────────────────────────

eco = pd.read_csv("../data/final/final_eco_2025.csv")
eco = eco.drop_duplicates(subset=["task_normalized", "title_current"]).copy()
print(f"Rows after dedup on (task_normalized, title_current): {len(eco):,}")
print(f"Unique tasks: {eco['task_normalized'].nunique():,}")
print(f"Unique occupations: {eco['title_current'].nunique():,}")

rating_cols = ["freq_mean", "importance", "relevance"]
eco[rating_cols] = eco[rating_cols].apply(pd.to_numeric, errors="coerce")

# Drop rows missing any rating
eco_clean = eco.dropna(subset=rating_cols).copy()
print(f"Rows with all three ratings: {len(eco_clean):,} ({len(eco_clean)/len(eco)*100:.1f}%)")

# ── 1. Global distributions ────────────────────────────────────────────────────

print("\n" + "="*60)
print("1. GLOBAL DISTRIBUTIONS")
print("="*60)

for col in rating_cols:
    s = eco_clean[col]
    print(f"\n{col}:")
    print(f"  mean={s.mean():.3f}  std={s.std():.3f}  median={s.median():.3f}")
    print(f"  min={s.min():.3f}  p25={s.quantile(.25):.3f}  p75={s.quantile(.75):.3f}  max={s.max():.3f}")

# ── 2. Global pairwise correlations ────────────────────────────────────────────

print("\n" + "="*60)
print("2. GLOBAL PAIRWISE CORRELATIONS")
print("="*60)

pairs = [("freq_mean", "importance"), ("freq_mean", "relevance"), ("importance", "relevance")]

for a, b in pairs:
    pearson_r, pearson_p = stats.pearsonr(eco_clean[a], eco_clean[b])
    spearman_r, spearman_p = stats.spearmanr(eco_clean[a], eco_clean[b])
    print(f"\n{a} vs {b}:")
    print(f"  Pearson:  r={pearson_r:.4f}  p={pearson_p:.2e}")
    print(f"  Spearman: r={spearman_r:.4f}  p={spearman_p:.2e}")

print("\nCorrelation matrix (Pearson):")
print(eco_clean[rating_cols].corr().round(4).to_string())

print("\nCorrelation matrix (Spearman):")
print(eco_clean[rating_cols].corr(method="spearman").round(4).to_string())

# ── 3. Within-occupation correlations ──────────────────────────────────────────

print("\n" + "="*60)
print("3. WITHIN-OCCUPATION CORRELATIONS")
print("="*60)
print("(Pearson r per occupation, then distribution of those r values)")

occ_corrs = {}
for a, b in pairs:
    rs = []
    for title, grp in eco_clean.groupby("title_current"):
        if len(grp) < 5:  # need enough tasks for meaningful correlation
            continue
        if grp[a].std() == 0 or grp[b].std() == 0:
            continue
        r, _ = stats.pearsonr(grp[a], grp[b])
        rs.append(r)
    rs = pd.Series(rs)
    occ_corrs[(a, b)] = rs
    print(f"\n{a} vs {b} (n={len(rs)} occs with 5+ tasks):")
    print(f"  mean r={rs.mean():.4f}  median r={rs.median():.4f}  std={rs.std():.4f}")
    print(f"  % positive: {(rs > 0).mean()*100:.1f}%")
    print(f"  % |r| > 0.5: {(rs.abs() > 0.5).mean()*100:.1f}%")

# ── 4. Within-occupation coefficient of variation ──────────────────────────────

print("\n" + "="*60)
print("4. WITHIN-OCCUPATION VARIATION (CV = std/mean)")
print("="*60)
print("How much do ratings vary within a single occupation?")

occ_cvs = []
for title, grp in eco_clean.groupby("title_current"):
    if len(grp) < 3:
        continue
    row = {"title_current": title, "n_tasks": len(grp)}
    for col in rating_cols:
        m = grp[col].mean()
        row[f"{col}_cv"] = grp[col].std() / m if m > 0 else np.nan
        row[f"{col}_mean"] = m
        row[f"{col}_std"] = grp[col].std()
    occ_cvs.append(row)

cv_df = pd.DataFrame(occ_cvs)
for col in rating_cols:
    cv_col = f"{col}_cv"
    s = cv_df[cv_col].dropna()
    print(f"\n{col} CV across occupations:")
    print(f"  mean={s.mean():.3f}  median={s.median():.3f}  std={s.std():.3f}")
    print(f"  p10={s.quantile(.1):.3f}  p90={s.quantile(.9):.3f}")

# ── 5. Task-level: do high-freq tasks have high importance? ────────────────────

print("\n" + "="*60)
print("5. FREQ BINS × IMPORTANCE/RELEVANCE")
print("="*60)

eco_clean["freq_bin"] = pd.cut(eco_clean["freq_mean"], bins=5, labels=["very low", "low", "medium", "high", "very high"])
bin_stats = eco_clean.groupby("freq_bin", observed=True).agg(
    n=("freq_mean", "count"),
    freq_mean=("freq_mean", "mean"),
    importance_mean=("importance", "mean"),
    importance_std=("importance", "std"),
    relevance_mean=("relevance", "mean"),
    relevance_std=("relevance", "std"),
).round(3)
print(bin_stats.to_string())

# ── 6. Composite weight candidates ────────────────────────────────────────────

print("\n" + "="*60)
print("6. COMPOSITE WEIGHT CANDIDATES")
print("="*60)
print("Testing different task weighting schemes and their distributions")

eco_clean["w_freq"] = eco_clean["freq_mean"]
eco_clean["w_imp_rel"] = eco_clean["importance"] * eco_clean["relevance"] / 100
eco_clean["w_freq_imp"] = eco_clean["freq_mean"] * eco_clean["importance"]
eco_clean["w_freq_imp_rel"] = eco_clean["freq_mean"] * eco_clean["importance"] * eco_clean["relevance"] / 100
eco_clean["w_rel_2imp"] = eco_clean["relevance"] * (2 ** eco_clean["importance"])  # the imp method from dashboard
eco_clean["w_freq_imp2_rel"] = eco_clean["freq_mean"] * (eco_clean["importance"] ** 2) * eco_clean["relevance"] / 100

# Then add "w_freq_imp2_rel" to weight_cols and re-run sections 6 and 8

weight_cols = ["w_freq", "w_imp_rel", "w_freq_imp", "w_freq_imp_rel", "w_rel_2imp", "w_freq_imp2_rel"]

for w in weight_cols:
    s = eco_clean[w]
    print(f"\n{w}:")
    print(f"  mean={s.mean():.3f}  std={s.std():.3f}  cv={s.std()/s.mean():.3f}")
    print(f"  min={s.min():.3f}  p25={s.quantile(.25):.3f}  median={s.median():.3f}  p75={s.quantile(.75):.3f}  max={s.max():.3f}")

# Correlation between weight schemes
print("\nCorrelation between weight schemes:")
print(eco_clean[weight_cols].corr().round(4).to_string())

# ── 7. What would wage allocation look like? ──────────────────────────────────

print("\n" + "="*60)
print("7. WAGE ALLOCATION EXAMPLES")
print("="*60)
print("For a sample occupation, show how different weights split the wage")

# Pick an occupation with a decent number of tasks
sample_occ = eco_clean.groupby("title_current").size()
sample_occ = sample_occ[(sample_occ >= 10) & (sample_occ <= 20)].index[5]  # arbitrary pick

sample = eco_clean[eco_clean["title_current"] == sample_occ].copy()
wage = sample["a_med_nat_2024"].iloc[0]
n_tasks = len(sample)

print(f"\nOccupation: {sample_occ}")
print(f"Median wage: ${wage:,.0f}")
print(f"Tasks: {n_tasks}")
print(f"Equal split per task: ${wage/n_tasks:,.0f}")

print(f"\n{'Task (truncated)':<50} {'Freq':>6} {'Imp':>5} {'Rel':>5} | {'Equal':>8} {'Freq':>8} {'Imp×Rel':>8} {'F×I×R':>8}")
print("-" * 130)

for w in ["w_freq", "w_imp_rel", "w_freq_imp_rel","w_rel_2imp", "w_freq_imp2_rel"]:
    total_w = sample[w].sum()
    sample[f"wage_{w}"] = (sample[w] / total_w) * wage if total_w > 0 else wage / n_tasks

for _, row in sample.iterrows():
    task_short = row["task_normalized"][:48]
    print(f"{task_short:<50} {row['freq_mean']:>6.2f} {row['importance']:>5.1f} {row['relevance']:>5.0f} | "
          f"${wage/n_tasks:>7,.0f} ${row['wage_w_freq']:>7,.0f} ${row['wage_w_imp_rel']:>7,.0f} ${row['wage_w_freq_imp_rel']:>7,.0f}")

# ── 8. Gini coefficient of weights within occupations ─────────────────────────

print("\n" + "="*60)
print("8. INEQUALITY OF WEIGHT DISTRIBUTIONS WITHIN OCCUPATIONS")
print("="*60)
print("Gini coefficient: 0 = perfectly equal, 1 = one task gets everything")

def gini(arr):
    arr = np.sort(np.array(arr, dtype=float))
    n = len(arr)
    if n == 0 or arr.sum() == 0:
        return 0
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * arr) - (n + 1) * np.sum(arr)) / (n * np.sum(arr))

occ_ginis = []
for title, grp in eco_clean.groupby("title_current"):
    if len(grp) < 5:
        continue
    row = {"title_current": title, "n_tasks": len(grp)}
    row["gini_equal"] = 0.0  # baseline
    for w in weight_cols:
        row[f"gini_{w}"] = gini(grp[w].values)
    occ_ginis.append(row)

gini_df = pd.DataFrame(occ_ginis)
for w in weight_cols:
    g = gini_df[f"gini_{w}"]
    print(f"\n{w}:")
    print(f"  mean gini={g.mean():.4f}  median={g.median():.4f}  std={g.std():.4f}")
    print(f"  p10={g.quantile(.1):.4f}  p90={g.quantile(.9):.4f}")

print("\n\nDone.")

Rows after dedup on (task_normalized, title_current): 18,796
Unique tasks: 17,507
Unique occupations: 923
Rows with all three ratings: 18,796 (100.0%)

1. GLOBAL DISTRIBUTIONS

freq_mean:
  mean=1.608  std=1.480  median=1.120
  min=0.004  p25=0.468  p75=2.335  max=7.858

importance:
  mean=3.995  std=0.482  median=4.040
  min=1.440  p25=3.700  p75=4.350  max=5.000

relevance:
  mean=80.870  std=18.955  median=86.670
  min=25.000  p25=70.010  p75=96.240  max=100.000

2. GLOBAL PAIRWISE CORRELATIONS

freq_mean vs importance:
  Pearson:  r=0.6033  p=0.00e+00
  Spearman: r=0.6364  p=0.00e+00

freq_mean vs relevance:
  Pearson:  r=0.1416  p=7.92e-85
  Spearman: r=0.1385  p=3.61e-81

importance vs relevance:
  Pearson:  r=0.4175  p=0.00e+00
  Spearman: r=0.4340  p=0.00e+00

Correlation matrix (Pearson):
            freq_mean  importance  relevance
freq_mean      1.0000      0.6033     0.1416
importance     0.6033      1.0000     0.4175
relevance      0.1416      0.4175     1.0000

Correlatio

In [10]:
"""
Task Rating Correlation Analysis
Goal: Understand relationships between freq_mean, importance, and relevance
to inform how to split occupation-level wages and employment across tasks.
"""

import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path

# ── Load and dedup ──────────────────────────────────────────────────────────────

eco = pd.read_csv("../data/final/final_eco_2025.csv")
eco = eco.drop_duplicates(subset=["task_normalized", "title_current"]).copy()
print(f"Rows after dedup on (task_normalized, title_current): {len(eco):,}")
print(f"Unique tasks: {eco['task_normalized'].nunique():,}")
print(f"Unique occupations: {eco['title_current'].nunique():,}")

rating_cols = ["freq_mean", "importance", "relevance"]
eco[rating_cols] = eco[rating_cols].apply(pd.to_numeric, errors="coerce")

# Drop rows missing any rating
eco_clean = eco.dropna(subset=rating_cols).copy()
print(f"Rows with all three ratings: {len(eco_clean):,} ({len(eco_clean)/len(eco)*100:.1f}%)")

# ── 1. Global distributions ────────────────────────────────────────────────────

print("\n" + "="*60)
print("1. GLOBAL DISTRIBUTIONS")
print("="*60)

for col in rating_cols:
    s = eco_clean[col]
    print(f"\n{col}:")
    print(f"  mean={s.mean():.3f}  std={s.std():.3f}  median={s.median():.3f}")
    print(f"  min={s.min():.3f}  p25={s.quantile(.25):.3f}  p75={s.quantile(.75):.3f}  max={s.max():.3f}")

# ── 2. Global pairwise correlations ────────────────────────────────────────────

print("\n" + "="*60)
print("2. GLOBAL PAIRWISE CORRELATIONS")
print("="*60)

pairs = [("freq_mean", "importance"), ("freq_mean", "relevance"), ("importance", "relevance")]

for a, b in pairs:
    pearson_r, pearson_p = stats.pearsonr(eco_clean[a], eco_clean[b])
    spearman_r, spearman_p = stats.spearmanr(eco_clean[a], eco_clean[b])
    print(f"\n{a} vs {b}:")
    print(f"  Pearson:  r={pearson_r:.4f}  p={pearson_p:.2e}")
    print(f"  Spearman: r={spearman_r:.4f}  p={spearman_p:.2e}")

print("\nCorrelation matrix (Pearson):")
print(eco_clean[rating_cols].corr().round(4).to_string())

# ── 3. Within-occupation correlations ──────────────────────────────────────────

print("\n" + "="*60)
print("3. WITHIN-OCCUPATION CORRELATIONS")
print("="*60)

occ_corrs = {}
for a, b in pairs:
    rs = []
    for title, grp in eco_clean.groupby("title_current"):
        if len(grp) < 5:
            continue
        if grp[a].std() == 0 or grp[b].std() == 0:
            continue
        r, _ = stats.pearsonr(grp[a], grp[b])
        rs.append(r)
    rs = pd.Series(rs)
    occ_corrs[(a, b)] = rs
    print(f"\n{a} vs {b} (n={len(rs)} occs):")
    print(f"  mean r={rs.mean():.4f}  median r={rs.median():.4f}")

# ── 4. Within-occupation variation ──────────────────────────────────────────────

print("\n" + "="*60)
print("4. WITHIN-OCCUPATION VARIATION (CV = std/mean)")
print("="*60)

occ_cvs = []
for title, grp in eco_clean.groupby("title_current"):
    if len(grp) < 3:
        continue
    row = {"title_current": title}
    for col in rating_cols:
        m = grp[col].mean()
        row[f"{col}_cv"] = grp[col].std() / m if m > 0 else np.nan
    occ_cvs.append(row)

cv_df = pd.DataFrame(occ_cvs)
for col in rating_cols:
    s = cv_df[f"{col}_cv"].dropna()
    print(f"\n{col} CV: mean={s.mean():.3f}  median={s.median():.3f}")

# ── 5. Freq Bins ──────────────────────────────────────────────────────────────

eco_clean["freq_bin"] = pd.cut(eco_clean["freq_mean"], bins=5, labels=["very low", "low", "medium", "high", "very high"])
bin_stats = eco_clean.groupby("freq_bin", observed=True).agg(
    n=("freq_mean", "count"),
    importance_mean=("importance", "mean"),
    relevance_mean=("relevance", "mean"),
).round(3)
print("\n" + "="*60)
print("5. FREQ BINS × IMPORTANCE/RELEVANCE")
print("="*60)
print(bin_stats.to_string())

# ── 6. Composite weight candidates ────────────────────────────────────────────

print("\n" + "="*60)
print("6. COMPOSITE WEIGHT CANDIDATES")
print("="*60)

eco_clean["w_freq"] = eco_clean["freq_mean"]
eco_clean["w_imp_rel"] = eco_clean["importance"] * eco_clean["relevance"] / 100
eco_clean["w_freq_imp_rel"] = eco_clean["freq_mean"] * eco_clean["importance"] * eco_clean["relevance"] / 100
eco_clean["w_rel_2imp"] = eco_clean["relevance"] * (2 ** eco_clean["importance"])
eco_clean["w_imp2_rel"] = (eco_clean["importance"] ** 2) * eco_clean["relevance"] / 100
eco_clean["w_freq_imp2_rel"] = eco_clean["freq_mean"] * (eco_clean["importance"] ** 2) * eco_clean["relevance"] / 100
eco_clean["w_freq_2imp_rel"] = eco_clean["freq_mean"] * (2 ** eco_clean["importance"]) * eco_clean["relevance"] / 100

weight_cols = ["w_freq", "w_imp_rel", "w_freq_imp_rel", "w_rel_2imp", "w_imp2_rel", "w_freq_imp2_rel", "w_freq_2imp_rel"]

for w in weight_cols:
    s = eco_clean[w]
    print(f"\n{w}: mean={s.mean():.3f}  median={s.median():.3f}  cv={s.std()/s.mean():.3f}")

# ── 7. Allocation Examples (Wage & Employment) ────────────────────────────────

print("\n" + "="*60)
print("7. WAGE & EMPLOYMENT ALLOCATION EXAMPLES")
print("="*60)

sample_occ = eco_clean.groupby("title_current").size()
sample_occ = sample_occ[(sample_occ >= 10) & (sample_occ <= 20)].index[5]

sample = eco_clean[eco_clean["title_current"] == sample_occ].copy()
wage = sample["a_med_nat_2024"].iloc[0]
emp = sample["emp_tot_nat_2024"].iloc[0]
n_tasks = len(sample)

print(f"\nOccupation: {sample_occ}")
print(f"Median Wage: ${wage:,.0f} | Total Employment: {emp:,.0f}")
print(f"Tasks: {n_tasks}")

# Header
cols = ["Freq", "Imp×Rel", "F×I×R", "Rel×2^I", "I²×Rel", "F×I²×R", "F×2^I×R"]
print(f"\n{'Task (truncated)':<40} | {'Wage Allocation ($)':^65} | {'Emp Allocation (n)':^55}")
print(f"{'':<40} | " + " ".join([f"{c:>10}" for c in cols]) + " | " + " ".join([f"{c:>8}" for c in cols]))
print("-" * 175)

for w in weight_cols:
    total_w = sample[w].sum()
    sample[f"wage_{w}"] = (sample[w] / total_w) * wage if total_w > 0 else wage / n_tasks
    sample[f"emp_{w}"] = (sample[w] / total_w) * emp if total_w > 0 else emp / n_tasks

for _, row in sample.iterrows():
    task_short = row["task_normalized"][:38]
    wages = " ".join([f"{row[f'wage_{w}']:>10,.0f}" for w in weight_cols])
    emps = " ".join([f"{row[f'emp_{w}']:>8,.0f}" for w in weight_cols])
    print(f"{task_short:<40} | {wages} | {emps}")

# ── 8. Gini coefficient ────────────────────────────────────────────────────────

print("\n" + "="*60)
print("8. INEQUALITY OF WEIGHT DISTRIBUTIONS (GINI)")
print("="*60)

def gini(arr):
    arr = np.sort(np.array(arr, dtype=float))
    n = len(arr)
    if n == 0 or arr.sum() == 0: return 0
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * arr) - (n + 1) * np.sum(arr)) / (n * np.sum(arr))

occ_ginis = []
for title, grp in eco_clean.groupby("title_current"):
    if len(grp) < 5: continue
    row = {"title_current": title}
    for w in weight_cols:
        row[f"gini_{w}"] = gini(grp[w].values)
    occ_ginis.append(row)

gini_df = pd.DataFrame(occ_ginis)
for w in weight_cols:
    print(f"{w:^15}: mean gini={gini_df[f'gini_{w}'].mean():.4f}")

print("\nDone.")

Rows after dedup on (task_normalized, title_current): 18,796
Unique tasks: 17,507
Unique occupations: 923
Rows with all three ratings: 18,796 (100.0%)

1. GLOBAL DISTRIBUTIONS

freq_mean:
  mean=1.608  std=1.480  median=1.120
  min=0.004  p25=0.468  p75=2.335  max=7.858

importance:
  mean=3.995  std=0.482  median=4.040
  min=1.440  p25=3.700  p75=4.350  max=5.000

relevance:
  mean=80.870  std=18.955  median=86.670
  min=25.000  p25=70.010  p75=96.240  max=100.000

2. GLOBAL PAIRWISE CORRELATIONS

freq_mean vs importance:
  Pearson:  r=0.6033  p=0.00e+00
  Spearman: r=0.6364  p=0.00e+00

freq_mean vs relevance:
  Pearson:  r=0.1416  p=7.92e-85
  Spearman: r=0.1385  p=3.61e-81

importance vs relevance:
  Pearson:  r=0.4175  p=0.00e+00
  Spearman: r=0.4340  p=0.00e+00

Correlation matrix (Pearson):
            freq_mean  importance  relevance
freq_mean      1.0000      0.6033     0.1416
importance     0.6033      1.0000     0.4175
relevance      0.1416      0.4175     1.0000

3. WITHIN-

## State Imputation Numbers

In [8]:
### State-Level Imputation Analysis (2024 OEWS) ###

# Load the data
oews_states = pd.read_csv("../data/oews_states_2024.csv")
title_soc = pd.read_csv("../data/merged_data_files/title_and_2019_soc.csv")
our_soc_codes = title_soc["soc_code_2019"].unique()
n_soc = len(our_soc_codes)

# Build state mapping
state_map = (
    oews_states[["AREA_TITLE", "PRIM_STATE"]]
    .drop_duplicates(subset="AREA_TITLE")
    .set_index("AREA_TITLE")["PRIM_STATE"]
    .to_dict()
)

# Convert wage/emp columns to numeric
oews_states["H_MEDIAN"] = pd.to_numeric(oews_states["H_MEDIAN"], errors="coerce")
oews_states["A_MEDIAN"] = pd.to_numeric(oews_states["A_MEDIAN"], errors="coerce")
oews_states["TOT_EMP"] = pd.to_numeric(oews_states["TOT_EMP"], errors="coerce")

results = []
for state_name, abbrev in state_map.items():
    state_data = oews_states[oews_states["AREA_TITLE"] == state_name].copy()
    matched = state_data[state_data["OCC_CODE"].isin(our_soc_codes)]
    n_matched = matched["OCC_CODE"].nunique()
    n_no_match = n_soc - n_matched

    both_wage_missing = matched["H_MEDIAN"].isna() & matched["A_MEDIAN"].isna()
    h_only = matched["H_MEDIAN"].notna() & matched["A_MEDIAN"].isna()
    a_only = matched["H_MEDIAN"].isna() & matched["A_MEDIAN"].notna()
    emp_missing = matched["TOT_EMP"].isna()

    wage_total_imputed = n_no_match + int(both_wage_missing.sum()) + int(h_only.sum()) + int(a_only.sum())
    emp_total_imputed = n_no_match + int(emp_missing.sum())

    results.append({
        "state": abbrev,
        "soc_in_pipeline": n_soc,
        "matched_in_state": n_matched,
        "no_state_data": n_no_match,
        "no_state_data_pct": round(n_no_match / n_soc * 100, 1),
        "wage_h_to_a": int(h_only.sum()),
        "wage_a_to_h": int(a_only.sum()),
        "wage_both_missing": int(both_wage_missing.sum()),
        "wage_total_imputed": wage_total_imputed,
        "wage_imputed_pct": round(wage_total_imputed / n_soc * 100, 1),
        "emp_total_imputed": emp_total_imputed,
        "emp_imputed_pct": round(emp_total_imputed / n_soc * 100, 1),
    })

imputation_df = pd.DataFrame(results).sort_values("state").reset_index(drop=True)

# --- Summary stats across all states ---
n_states = len(imputation_df)
avg_wage_pct = imputation_df["wage_imputed_pct"].mean()
med_wage_pct = imputation_df["wage_imputed_pct"].median()
min_wage_pct = imputation_df["wage_imputed_pct"].min()
max_wage_pct = imputation_df["wage_imputed_pct"].max()
min_wage_st = imputation_df.loc[imputation_df["wage_imputed_pct"].idxmin(), "state"]
max_wage_st = imputation_df.loc[imputation_df["wage_imputed_pct"].idxmax(), "state"]

avg_emp_pct = imputation_df["emp_imputed_pct"].mean()
med_emp_pct = imputation_df["emp_imputed_pct"].median()
min_emp_pct = imputation_df["emp_imputed_pct"].min()
max_emp_pct = imputation_df["emp_imputed_pct"].max()
min_emp_st = imputation_df.loc[imputation_df["emp_imputed_pct"].idxmin(), "state"]
max_emp_st = imputation_df.loc[imputation_df["emp_imputed_pct"].idxmax(), "state"]

avg_no_data_pct = imputation_df["no_state_data_pct"].mean()
avg_h_to_a = imputation_df["wage_h_to_a"].mean()
avg_a_to_h = imputation_df["wage_a_to_h"].mean()
avg_both_missing = imputation_df["wage_both_missing"].mean()

print(f"Total unique SOC codes in pipeline: {n_soc}")
print(f"States/territories analyzed: {n_states}")
print()

# --- Copy-paste blurb for paper ---
lines = [
    f"Across {n_states} states and territories, an average of {avg_wage_pct:.1f}% of the {n_soc} occupation-level SOC codes required some form of wage imputation (median: {med_wage_pct:.1f}%, range: {min_wage_pct:.1f}% [{min_wage_st}] to {max_wage_pct:.1f}% [{max_wage_st}]).",
    "",
    f"Of these, an average of {avg_no_data_pct:.1f}% of SOC codes had no state-level data and were assigned national median wages. An average of {avg_h_to_a:.1f} codes per state had hourly but not annual wages (imputed as hourly x 2,080), {avg_a_to_h:.1f} had annual but not hourly (imputed as annual / 2,080), and {avg_both_missing:.1f} had neither (assigned national values).",
    "",
    f"For employment, an average of {avg_emp_pct:.1f}% of SOC codes required imputation (median: {med_emp_pct:.1f}%, range: {min_emp_pct:.1f}% [{min_emp_st}] to {max_emp_pct:.1f}% [{max_emp_st}]), where missing state employment was estimated as the national figure scaled by the state's share of total national employment.",
]
print("--- COPY-PASTE FOR PAPER (ALL STATES SUMMARY) ---")
print("\n".join(lines))
print()

# --- Specific state/territory breakdowns ---
highlight_states = ["UT", "CA", "DC", "GU", "PR", "VI"]
highlight_df = imputation_df[imputation_df["state"].isin(highlight_states)].copy()
highlight_df = highlight_df.set_index("state").loc[highlight_states].reset_index()

print("--- SPECIFIC STATE/TERRITORY BREAKDOWN ---")
display(highlight_df)
print()

print("--- COPY-PASTE FOR PAPER (SPECIFIC STATES) ---")
print()
for _, row in highlight_df.iterrows():
    st = row["state"]
    lines = [
        f"{st}:",
        f"  Of {row['soc_in_pipeline']} SOC codes, {row['matched_in_state']} ({100 - row['no_state_data_pct']:.1f}%) had direct state-level data.",
        f"  Wage imputation: {row['wage_total_imputed']} codes ({row['wage_imputed_pct']:.1f}%) required imputation -- {row['no_state_data']} had no state data (used national), {row['wage_h_to_a']} had hourly only (annual = hourly x 2,080), {row['wage_a_to_h']} had annual only (hourly = annual / 2,080), and {row['wage_both_missing']} had neither (used national).",
        f"  Employment imputation: {row['emp_total_imputed']} codes ({row['emp_imputed_pct']:.1f}%) required imputation (national employment x state share).",
        "",
    ]
    print("\n".join(lines))

display(imputation_df)


Total unique SOC codes in pipeline: 617
States/territories analyzed: 54

--- COPY-PASTE FOR PAPER (ALL STATES SUMMARY) ---
Across 54 states and territories, an average of 28.7% of the 617 occupation-level SOC codes required some form of wage imputation (median: 24.7%, range: 16.4% [NY] to 78.9% [VI]).

Of these, an average of 19.3% of SOC codes had no state-level data and were assigned national median wages. An average of 1.6 codes per state had hourly but not annual wages (imputed as hourly x 2,080), 43.1 had annual but not hourly (imputed as annual / 2,080), and 13.5 had neither (assigned national values).

For employment, an average of 21.9% of SOC codes required imputation (median: 17.3%, range: 8.8% [PA] to 78.0% [VI]), where missing state employment was estimated as the national figure scaled by the state's share of total national employment.

--- SPECIFIC STATE/TERRITORY BREAKDOWN ---


,state,soc_in_pipeline,matched_in_state,no_state_data,no_state_data_pct,wage_h_to_a,wage_a_to_h,wage_both_missing,wage_total_imputed,wage_imputed_pct,emp_total_imputed,emp_imputed_pct
0,UT,617,536,81,13.1,3,51,15,150,24.3,93,15.1
1,CA,617,575,42,6.8,1,50,16,109,17.7,58,9.4
2,DC,617,413,204,33.1,0,39,17,260,42.1,236,38.2
3,GU,617,169,448,72.6,1,4,6,459,74.4,453,73.4
4,PR,617,431,186,30.1,1,37,5,229,37.1,201,32.6
5,VI,617,136,481,78.0,0,3,3,487,78.9,481,78.0



--- COPY-PASTE FOR PAPER (SPECIFIC STATES) ---

UT:
  Of 617 SOC codes, 536 (86.9%) had direct state-level data.
  Wage imputation: 150 codes (24.3%) required imputation -- 81 had no state data (used national), 3 had hourly only (annual = hourly x 2,080), 51 had annual only (hourly = annual / 2,080), and 15 had neither (used national).
  Employment imputation: 93 codes (15.1%) required imputation (national employment x state share).

CA:
  Of 617 SOC codes, 575 (93.2%) had direct state-level data.
  Wage imputation: 109 codes (17.7%) required imputation -- 42 had no state data (used national), 1 had hourly only (annual = hourly x 2,080), 50 had annual only (hourly = annual / 2,080), and 16 had neither (used national).
  Employment imputation: 58 codes (9.4%) required imputation (national employment x state share).

DC:
  Of 617 SOC codes, 413 (66.9%) had direct state-level data.
  Wage imputation: 260 codes (42.1%) required imputation -- 204 had no state data (used national), 0 had ho

,state,soc_in_pipeline,matched_in_state,no_state_data,no_state_data_pct,wage_h_to_a,wage_a_to_h,wage_both_missing,wage_total_imputed,wage_imputed_pct,emp_total_imputed,emp_imputed_pct
0,AK,617,415,202,32.7,0,32,14,248,40.2,220,35.7
1,AL,617,524,93,15.1,2,47,11,153,24.8,107,17.3
2,AR,617,492,125,20.3,2,46,13,186,30.1,136,22.0
3,AZ,617,534,83,13.5,2,50,14,149,24.1,107,17.3
4,CA,617,575,42,6.8,1,50,16,109,17.7,58,9.4
5,CO,617,535,82,13.3,0,47,20,149,24.1,102,16.5
6,CT,617,518,99,16.0,2,48,17,166,26.9,122,19.8
7,DC,617,413,204,33.1,0,39,17,260,42.1,236,38.2
8,DE,617,421,196,31.8,0,32,24,252,40.8,223,36.1
9,FL,617,568,49,7.9,3,49,14,115,18.6,70,11.3


## Imputation Numbers For DWS Ratings

In [9]:
import pandas as pd
import re
from pathlib import Path

DATA_DIR = Path("../data")

# ── 1. Load datasets ────────────────────────────────────────────
eco_2025 = pd.read_csv(DATA_DIR / "final" / "final_eco_2025.csv")
dws = pd.read_csv(DATA_DIR / "dws_ratings.csv")

# ── 2. Normalize titles for matching ────────────────────────────
def normalize_title(text):
    if not isinstance(text, str):
        return text
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

eco_2025["title_current_norm"] = eco_2025["title_current"].apply(normalize_title)
dws["title_norm"] = dws["Occupation Title"].apply(normalize_title)

# ── 3. Build DWS lookup (normalized title -> Star Ratings) ─────
dws_lookup = dws[["title_norm", "Star Ratings"]].drop_duplicates(subset="title_norm")
dws_lookup = dws_lookup.rename(columns={"Star Ratings": "dws_star_rating"})

# ── 4. Merge star ratings into eco 2025 ─────────────────────────
eco_2025 = eco_2025.merge(dws_lookup, left_on="title_current_norm", right_on="title_norm", how="left")
eco_2025.drop(columns=["title_current_norm", "title_norm"], inplace=True)

# ── 5. Build occ-level df for imputation ───────────────────────
occ_ratings = (
    eco_2025[["title_current", "broad_occ", "minor_occ_category", "major_occ_category", "dws_star_rating"]]
    .drop_duplicates(subset="title_current")
    .sort_values("title_current")
    .reset_index(drop=True)
)

direct = occ_ratings["dws_star_rating"].notna().sum()
print(f"Direct match: {direct}/{len(occ_ratings)} occupations")

# Track imputation source
occ_ratings["dws_rating_source"] = occ_ratings["dws_star_rating"].apply(
    lambda x: "direct" if pd.notna(x) else None
)

# ── 6. Impute: broad_occ -> minor_occ_category -> major_occ_category
for level_col, level_name in [
    ("broad_occ", "broad"),
    ("minor_occ_category", "minor"),
    ("major_occ_category", "major"),
]:
    # Compute mean star rating from direct-matched occs at this level
    known = occ_ratings[occ_ratings["dws_star_rating"].notna()]
    level_means = (
        known.groupby(level_col)["dws_star_rating"]
        .mean()
        .round()
        .astype(int)
        .to_dict()
    )

    missing_mask = occ_ratings["dws_star_rating"].isna()
    filled = occ_ratings.loc[missing_mask, level_col].map(level_means)
    fill_mask = missing_mask & filled.notna()

    occ_ratings.loc[fill_mask, "dws_star_rating"] = filled[fill_mask]
    occ_ratings.loc[fill_mask, "dws_rating_source"] = level_name

    n_filled = fill_mask.sum()
    remaining = occ_ratings["dws_star_rating"].isna().sum()
    print(f"After {level_name} imputation: +{n_filled} filled, {remaining} still missing")

occ_ratings["dws_star_rating"] = occ_ratings["dws_star_rating"].astype("Int64")

# ── 7. Summary ─────────────────────────────────────────────────
print()
print(f"Final: {occ_ratings['dws_star_rating'].notna().sum()}/{len(occ_ratings)} occupations rated")
print(occ_ratings["dws_rating_source"].value_counts())

occ_ratings


Direct match: 607/923 occupations
After broad imputation: +197 filled, 119 still missing
After minor imputation: +103 filled, 16 still missing
After major imputation: +16 filled, 0 still missing

Final: 923/923 occupations rated
dws_rating_source
direct    607
broad     197
minor     103
major      16
Name: count, dtype: int64


,title_current,broad_occ,minor_occ_category,major_occ_category,dws_star_rating,dws_rating_source
0,Accountants and Auditors,Accountants and Auditors,Financial Specialists,Business and Financial Operations Occupations,5,direct
1,Actors,"Actors, Producers, and Directors","Entertainers and Performers, Sports and Relate...","Arts, Design, Entertainment, Sports, and Media...",0,direct
2,Actuaries,Actuaries,Mathematical Science Occupations,Computer and Mathematical Occupations,4,direct
3,Acupuncturists,Miscellaneous Healthcare Diagnosing or Treatin...,Healthcare Diagnosing or Treating Practitioners,Healthcare Practitioners and Technical Occupat...,5,broad
4,Acute Care Nurses,Registered Nurses,Healthcare Diagnosing or Treating Practitioners,Healthcare Practitioners and Technical Occupat...,5,broad
...,...,...,...,...,...,...
918,Wind Turbine Service Technicians,Wind Turbine Service Technicians,"Other Installation, Maintenance, and Repair Oc...","Installation, Maintenance, and Repair Occupations",3,minor
919,"Woodworking Machine Setters, Operators, and Te...","Woodworking Machine Setters, Operators, and Te...",Woodworkers,Production Occupations,2,direct
920,Word Processors and Typists,Data Entry and Information Processing Workers,Other Office and Administrative Support Workers,Office and Administrative Support Occupations,1,direct
921,Writers and Authors,Writers and Editors,Media and Communication Workers,"Arts, Design, Entertainment, Sports, and Media...",3,direct


## How Many AEI Task Occs Map Onto Eco 2025

In [4]:
import pandas as pd
import re
from pathlib import Path

DATA_DIR = Path("../data/final")
AEI_SNAPSHOTS = [
    "final_aei_v1.csv", "final_aei_v2.csv", "final_aei_v3.csv", 
    "final_aei_v4.csv", "final_aei_v5.csv", "final_aei_api_v3.csv", 
    "final_aei_api_v4.csv", "final_aei_api_v5.csv"
]

def normalize_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return " ".join(text.split())

# 1. Load ECO 2025 Backbone
df_eco = pd.read_csv(DATA_DIR / "final_eco_2025.csv")

# Create two distinct "Truth Sets"
# Set A: Matching on the 2010 Title (Legacy)
legacy_pairs = set(zip(df_eco["title"], df_eco["task_normalized"]))
# Set B: Matching on the 2019 Title (Current)
current_pairs = set(zip(df_eco["title_current"], df_eco["task_normalized"]))

print(f"Baseline Stats:")
print(f" - Unique 2010 (Title, Task) pairs: {len(legacy_pairs):,}")
print(f" - Unique 2019 (TitleCurrent, Task) pairs: {len(current_pairs):,}\n")

results = []
for file_name in AEI_SNAPSHOTS:
    path = DATA_DIR / file_name
    if not path.exists(): continue
    
    df_aei = pd.read_csv(path)
    df_aei["task_normalized"] = df_aei["task"].apply(normalize_text)
    
    # Generate the pairs present in the AEI file
    # Note: AEI files generally only have the 'title' column (2010 SOC)
    aei_pairs = list(zip(df_aei["title"], df_aei["task_normalized"]))
    total_aei = len(aei_pairs)
    
    # Check against both truth sets
    matched_legacy = [p for p in aei_pairs if p in legacy_pairs]
    matched_current = [p for p in aei_pairs if p in current_pairs]
    
    results.append({
        "File": file_name,
        "Total AEI": total_aei,
        "Matched Legacy": len(matched_legacy),
        "Legacy %": round((len(matched_legacy)/total_aei)*100, 2),
        "Matched Current": len(matched_current),
        "Current %": round((len(matched_current)/total_aei)*100, 2)
    })

summary_df = pd.DataFrame(results)
print(summary_df.to_string(index=False))

Baseline Stats:
 - Unique 2010 (Title, Task) pairs: 18,652
 - Unique 2019 (TitleCurrent, Task) pairs: 18,796

                File  Total AEI  Matched Legacy  Legacy %  Matched Current  Current %
    final_aei_v1.csv       5481            4510     82.28             3984      72.69
    final_aei_v2.csv       5264            4348     82.60             3850      73.14
    final_aei_v3.csv       4278            3534     82.61             3152      73.68
    final_aei_v4.csv       5143            4264     82.91             3778      73.46
    final_aei_v5.csv       5217            4309     82.60             3823      73.28
final_aei_api_v3.csv       3325            2721     81.83             2412      72.54
final_aei_api_v4.csv       3585            2948     82.23             2617      73.00
final_aei_api_v5.csv       3737            3063     81.96             2732      73.11
